In [ ]:
"""This setup will connect our month 1 "expert librarian" with a new "web researcher"
using the ReAct (Reason + Act) framework."""

# setup and environment loading
import sys
import os
from dotenv import load_dotenv, find_dotenv

# reaching out to the root folder to find the .env
load_dotenv(find_dotenv())

# 2. Add the root directory to sys.path so we can import the 'month_1' folder
sys.path.append(os.path.abspath(".."))

print("Environment and Paths configured")

Environment and Paths configured


In [3]:
# llm and embedding initialization
"""We initialize the brain of the agent. we are going to use a powerful llm for the agent logic,
as it follows tool-calling instructions much better than smaller models."""

from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# initializing the LLM
llm = Groq(
    model= 'llama-3.3-70B-versatile',
    api_key= os.getenv('GROQ_API_KEY')
)

Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)

print("⚜️ LLM and Embeddings initialized.")

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚜️ LLM and Embeddings initialized.


In [ ]:
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from llama_index.core import StorageContext, VectorStoreIndex

mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))

# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)


# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

# 2. Creating the 'index' object, loading it using the available context
index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

In [14]:
if os.getenv('MONGO_URI'): 
    print(' the uri exists')

 the uri exists


In [ ]:
# importing the month 1 tool
# this is where i turn my entire month 1 project into a single tool

# going up two levels since thats where our month_1 is: week1 -> Month2 -> 4 MONTHS JOURNEY (Root)
projectRoot = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if projectRoot not in sys.path:
    sys.path.append(projectRoot)

# now it will see our 'month_1' ArithmeticError    
from month_1.retrieval import RetrievalEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata

# initializing month 1 pipeline
m1Logic = RetrievalEngine(index= index, local_reasoner= llm)
m1Engine = m1Logic.get_query_engine()

# wrapping it as a tool
pdf_tool = QueryEngineTool(
    query_engine= m1Engine,
    metadata= ToolMetadata(
        name = "apple_10k_expert",
        description= (
            "Use this tool ONLY for questions about Apple's 2023 financial data,"
            "including revenue, debt, risk factors, and balance sheets found in the 10-k."
        ),
    ),
)

print("Month 1 library loaded as a tool...")

Month 1 library loaded as a tool...


In [16]:
# the researcher (tavily API)
from llama_index.tools.tavily_research import TavilyToolSpec

# initializing tavily
tavilySpec = TavilyToolSpec(api_key= os.getenv("TAVILY_API_KEY"))
search_tool = tavilySpec.to_tool_list()[0]

print("Web search tool (Tavily) initialized.")

Web search tool (Tavily) initialized.


In [17]:
# assembling the ReAct agent
"""the ReAct agent uses a 'thought -> action -> observation' loop. it will read the tool 
descriptions and decide which one to use"""

from llama_index.core.agent import ReActAgent

# combining my librarian and the researcher
tools = [pdf_tool, search_tool]

# creating the agent
agent = ReActAgent(
    tools = tools,
    llm= llm,
    verbose= True # this allows us to see the Agent's thought process
    
)

print("Agent 'Manager' is online and ready.")

Agent 'Manager' is online and ready.


In [18]:
print(type(agent))

<class 'llama_index.core.agent.workflow.react_agent.ReActAgent'>


In [19]:
# now we do the multi-step routing test
# we use a complex query to test routing logic

query = (
    "Based on the 10-K, what was Apple's total revenue in 2023,"
    "and what is the current stock market sentiment regarding Apple today?"
)

response = await agent.run(user_msg= query)
print("\n-----FINAL RESPONSE-----")
print(response)

INFO:workflows.verbose:[tick] add: AgentWorkflowStartEvent(user_msg="Based on the 10-K, what was Apple's total revenue ...", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
INFO:workflows.verbose:[init_run:0] started from AgentWorkflowStartEvent
INFO:workflows.verbose:[init_run:0] complete with AgentInput
INFO:workflows.verbose:[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="Based on the 10-K, what was Apple's total revenue in 2023,and what is...
INFO:workflows.verbose:[setup_agent:0] started from AgentInput
INFO:workflows.verbose:[setup_agent:0] complete with AgentSetup
INFO:workflows.verbose:[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="Based on the 10-K, what was Apple's total revenue in 2023,and what is...
INFO:workflows.verbose:[run_agent_step:0] started from AgentS

Step-back concept: Apple financial performance 2023. 

This broader search query still focuses on Apple and the year 2023, but it encompasses more aspects of the company's financial situation, including revenue, profits, expenses, and other related metrics, rather than just the total revenue.
***Some parent nodes missing in docstore. Falling back to basic retrieval.
DEBUG: After reranking → 5 nodes kept


INFO:workflows.verbose:[parse_agent_output:0] complete with no result
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:workflows.verbose:[call_tool:0] complete with ToolCallResult
INFO:workflows.verbose:[tick] add: ToolCallResult(tool_name='apple_10k_expert', tool_kwargs={'input': 'Apple total revenue 2023'}, tool_id='dfec614f-0b78-48e6-b313-c5afeb804135', tool_output=ToolOutput(blocks=[TextBlock(block_type='...
INFO:workflows.verbose:[aggregate_tool_results:0] started from ToolCallResult
INFO:workflows.verbose:[aggregate_tool_results:0] complete with AgentInput
INFO:workflows.verbose:[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="Based on the 10-K, what was Apple's total revenue in 2023,and what is...
INFO:workflows.verbose:[setup_agent:0] started from AgentInput
INFO:workflows.verbose:[setup_agent:0] complete with AgentSetup
INFO:workflows.v


-----FINAL RESPONSE-----
Apple's total revenue in 2023 was $383,285 million. The current stock market sentiment regarding Apple today is Bullish, with an overall options flow sentiment based on delta 40-60 options capturing pure directional conviction. The current price of AAPL shares is $267.92, with a price-to-earnings ratio of 34.55 and currently yields dividends of 38.1%.


In [ ]:
# viering some sys evaluation and Audit logs
# reviewing the last step's source to ensure it didn't just guess

# Safety check for sources
if hasattr(response, 'sources') and response.sources:
    print("\n--- Tools Used ---")
    for source in response.sources:
        print(f"Tool: {source.tool_name}")
        print(f"Input: {source.raw_input}")
        print("-" * 30)
else:
    print("\n(No detailed sources available in this response)")


(No detailed sources available in this response)


In [20]:
# testing retrieval engine directly
# Test your retrieval engine directly
query = "What was Apple's total revenue in 2023?"
nodes, concept = m1Logic.expert_retrieve(query)

print(f"Concept generated: {concept}")
print(f"Nodes found: {len(nodes)}")

if len(nodes) > 0:
    print("Top Node Content:", nodes[0].node.get_content()[:200])
else:
    print("❌ ZERO NODES FOUND. Check your MongoDB connection or Reranker threshold.")

INFO:openai._base_client:Retrying request to /chat/completions in 0.476467 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Step-back concept: What were Apple's financial performance and key statistics in 2023!
***Some parent nodes missing in docstore. Falling back to basic retrieval.
DEBUG: After reranking → 5 nodes kept
Concept generated: What were Apple's financial performance and key statistics in 2023!
Nodes found: 5
Top Node Content: # Operating Expenses

Operating expenses for 2024, 2023 and 2022 were as follows (dollars in millions):

|                                     | 2024     | Change | 2023     | Change | 2022     |
| --
